<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/185-Embedding_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Sample texts
texts = [
    "광대버섯(Amanita phalloides)은 크고 인상적인 후성(위) 자실체(담자과체)를 가지고 있습니다.",
    "큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.",
    "A. phalloides, 일명 Death Cap은 알려진 모든 버섯 중에서 가장 독성이 강한 버섯 중 하나입니다.",
    "고양이 다리는 3개입니다.",
]

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# You might need to install sentence-transformers if you haven't already
# !pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer

# 1. Load a pre-trained embedding model
# You can choose different models, e.g., 'paraphrase-MiniLM-L6-v2', 'distiluse-base-multilingual-cased-v1'
# For Korean, 'sentence-transformers/snunlp-KR-SBERT-V40' could be a good choice, but it requires a different setup if not directly available.
# Let's use a multilingual model for broad compatibility.
model_name = 'sentence-transformers/distiluse-base-multilingual-cased-v1'
try:
    embedding_model = SentenceTransformer(model_name)
except Exception as e:
    print(f"Error loading model {model_name}: {e}")
    print("Attempting to download and load a common multilingual model...")
    model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
    embedding_model = SentenceTransformer(model_name)

print(f"Using embedding model: {model_name}")

# 2. Generate embeddings for the `texts` variable
text_embeddings = embedding_model.encode(texts, convert_to_tensor=True)

print("Embeddings generated:")
print(text_embeddings.shape)

# 3. Create a simple vector store (in-memory for this example)
# In a real application, you would use a dedicated vector database like FAISS, ChromaDB, Pinecone, etc.
vector_store = {
    "texts": texts,
    "embeddings": text_embeddings
}

print("\nVector store created with embeddings.")

# 4. Implement RAG logic
def retrieve_documents(query, top_k=2):
    query_embedding = embedding_model.encode([query], convert_to_tensor=True)
    similarities = cosine_similarity(query_embedding.cpu().numpy(), vector_store["embeddings"].cpu().numpy())

    # Get the indices of the top_k most similar documents
    top_indices = np.argsort(similarities[0])[-top_k:][::-1]

    retrieved_docs = [vector_store["texts"][i] for i in top_indices]
    return retrieved_docs

def generate_response_with_llm(query, retrieved_docs):
    # This is a placeholder for your LLM interaction.
    # You would typically send the query and retrieved_docs to an LLM API (e.g., OpenAI, Google Gemini, Anthropic, or a local model).

    context = "\n".join(retrieved_docs)
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer: Based on the provided context, "

    # Example of how you might call an LLM (using a hypothetical LLM_API)
    # from your_llm_library import LLM_API
    # llm_response = LLM_API.generate(prompt)
    # return llm_response

    return f"[LLM would generate a response here based on the following prompt and context]\nPrompt: {prompt}\nRetrieved documents: {retrieved_docs}"

# Example usage:
query = "버섯 중에서 가장 독성이 강한 것이 무엇인가요?"
retrieved_documents = retrieve_documents(query, top_k=2)
print(f"\nQuery: {query}")
print(f"Retrieved documents: {retrieved_documents}")

llm_response = generate_response_with_llm(query, retrieved_documents)
print(f"LLM Response: {llm_response}")

query_cat = "고양이는 다리가 몇 개인가요?"
retrieved_documents_cat = retrieve_documents(query_cat, top_k=1)
print(f"\nQuery: {query_cat}")
print(f"Retrieved documents: {retrieved_documents_cat}")

llm_response_cat = generate_response_with_llm(query_cat, retrieved_documents_cat)
print(f"LLM Response: {llm_response_cat}")

In [6]:
from google.colab import userdata
import openai
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
openai.api_key  = os.environ["OPENAI_API_KEY"]

In [12]:
import openai
import os # Import os to access environment variables

# Initialize the OpenAI client with the API key from environment variables
# The API key should be set in a previous cell (KtdAqcMSQFuE) using `os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')`
client = openai.OpenAI()

# --- Modified generate_response_with_llm function ---
def generate_response_with_llm(query, retrieved_docs):
    context = "\n".join(retrieved_docs)

    # Construct a more detailed prompt for the LLM
    system_message = "You are a helpful assistant that answers questions based on the provided context. If the answer is not in the context, state that you don't have enough information."
    user_message = f"Based on the following context, answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    try:
        # Use the new OpenAI API syntax
        response = client.chat.completions.create(
            model="gpt-3.5-turbo", # Or gpt-4, gpt-4o, etc., depending on your access
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            temperature=0.7, # Adjust creativity
            max_tokens=150   # Adjust response length
        )
        return response.choices[0].message.content.strip() # Accessing content is slightly different
    except Exception as e:
        return f"Error calling OpenAI API: {e}"

# --- Example Usage with the new LLM integration ---
# Make sure to run the previous cells (mdkDgmU--1BD and QcQOQSjDO5FH) first
# to define `texts`, `embedding_model`, `vector_store`, and `retrieve_documents`.

# Example usage with the previous query:
query_mushroom = "버섯 중에서 가장 독성이 강한 것이 무엇인가요?"
retrieved_documents_mushroom = retrieve_documents(query_mushroom, top_k=2)
print(f"\nQuery: {query_mushroom}")
print(f"Retrieved documents: {retrieved_documents_mushroom}")

llm_response_mushroom = generate_response_with_llm(query_mushroom, retrieved_documents_mushroom)
print(f"LLM Response (OpenAI): {llm_response_mushroom}")

# Example usage with the previous cat query:
query_cat_updated = "고양이는 다리가 몇 개인가요?"
retrieved_documents_cat_updated = retrieve_documents(query_cat_updated, top_k=1)
print(f"\nQuery: {query_cat_updated}")
print(f"Retrieved documents: {retrieved_documents_cat_updated}")

llm_response_cat_updated = generate_response_with_llm(query_cat_updated, retrieved_documents_cat_updated)
print(f"LLM Response (OpenAI): {llm_response_cat_updated}")


Query: 버섯 중에서 가장 독성이 강한 것이 무엇인가요?
Retrieved documents: ['A. phalloides, 일명 Death Cap은 알려진 모든 버섯 중에서 가장 독성이 강한 버섯 중 하나입니다.', '큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.']
LLM Response (OpenAI): A. phalloides, 일명 Death Cap

Query: 고양이는 다리가 몇 개인가요?
Retrieved documents: ['고양이 다리는 3개입니다.']
LLM Response (OpenAI): 고양이는 다리가 3개입니다.
